# Experiment 5: Flow-Based Generative Model

**Name:** Noora  
**Roll No:** 2301420015  
**Course:** B.Tech CSE (Data Science)

---

## Objective
To understand normalizing flows — a class of generative models that learn an invertible transformation between a simple distribution and a complex data distribution.

## Theory
**Normalizing Flows** transform a simple base distribution p(z) into a complex data distribution p(x) using an invertible function f:

```
x = f(z),   z = f⁻¹(x)
```

The change-of-variables formula gives us:
```
log p(x) = log p(z) - log|det(∂f/∂z)|
```

**Common transformations:**
- Exponential: x = exp(z)  (maps Normal → Log-Normal)
- Affine: x = az + b
- Sigmoid / Logit

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm, lognorm

sns.set(style='whitegrid', font_scale=1.2)
np.random.seed(42)

N = 2000

# --- Base Distribution: Standard Normal ---
z = np.random.normal(0, 1, N)

# --- Transformations ---
x_exp    = np.exp(z)               # Normal -> Log-Normal
x_affine = 3 * z + 5              # Affine shift
x_sq     = z**2 * np.sign(z)      # Non-linear (signed square)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Normalizing Flows: Transforming Distributions', fontsize=15, fontweight='bold')

# Base
sns.histplot(z, kde=True, ax=axes[0,0], color='steelblue', stat='density')
axes[0,0].set_title('Base: Standard Normal z ~ N(0,1)')
axes[0,0].set_xlabel('z')

# Exponential
sns.histplot(x_exp, kde=True, ax=axes[0,1], color='purple', stat='density')
axes[0,1].set_title('x = exp(z)  →  Log-Normal Distribution')
axes[0,1].set_xlabel('x')

# Affine
sns.histplot(x_affine, kde=True, ax=axes[1,0], color='green', stat='density')
axes[1,0].set_title('x = 3z + 5  →  Affine Transformation')
axes[1,0].set_xlabel('x')

# Non-linear
sns.histplot(x_sq, kde=True, ax=axes[1,1], color='orange', stat='density')
axes[1,1].set_title('x = z²·sign(z)  →  Non-linear Flow')
axes[1,1].set_xlabel('x')

plt.tight_layout()
plt.savefig('exp5_flows.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Invertibility Demo: Forward & Inverse ---
z_sample = np.array([0.0, 0.5, 1.0, -0.5, -1.0])

# Forward flow: z -> x
x_sample = np.exp(z_sample)

# Inverse flow: x -> z
z_recovered = np.log(x_sample)

print("Invertibility Check (exp/log flow):")
print(f"{'z':>8} {'x=exp(z)':>12} {'log(x) recovered':>18} {'Match':>8}")
for z_i, x_i, zr_i in zip(z_sample, x_sample, z_recovered):
    match = '✓' if abs(z_i - zr_i) < 1e-10 else '✗'
    print(f"{z_i:>8.2f} {x_i:>12.4f} {zr_i:>18.4f} {match:>8}")

# --- Log-likelihood under the flow ---
z_ll = np.random.normal(0, 1, N)
x_ll = np.exp(z_ll)

log_pz  = norm.logpdf(z_ll, 0, 1)
log_det = z_ll                             # log |df/dz| = log(exp(z)) = z
log_px  = log_pz - log_det                 # Change of variables formula

print(f"\nAverage log p(x): {np.mean(log_px):.4f}")
print(f"True lognormal avg log-likelihood: {np.mean(lognorm.logpdf(x_ll, s=1, scale=1)):.4f}")

## Result
- The exponential transformation successfully converts a Normal distribution into a Log-Normal distribution.
- The transformation is perfectly invertible — recovered z values match original z values exactly.
- Log-likelihood under the flow matches the true Log-Normal log-likelihood, validating the change-of-variables formula.

## Conclusion
Flow-based models learn invertible transformations between simple and complex distributions. Unlike GANs or VAEs, flows provide exact likelihood computation, making them theoretically appealing. Real-world examples include RealNVP, Glow, and NICE.